In [1]:
!pip install argopy xarray netCDF4 pandas numpy pyarrow openpyxl tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.2.1 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.4.0 which is incompat

In [2]:
from argopy import DataFetcher as ArgoDataFetcher
import argopy
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# Argo Core Profile Dataset
# Purpose: temperature, salinity, pressure/depth, time, lat, lon, QC
# ============================================================

argopy.set_options(src="erddap", mode="expert")

OUTPUT_DIR = Path("argo_profile_data/raw_core_argo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASINS = {
    "bob": {
        "name": "Bay_of_Bengal",
        "lon_min": 80,
        "lon_max": 98,
        "lat_min": 5,
        "lat_max": 22
    },
    "as": {
        "name": "Arabian_Sea",
        "lon_min": 60,
        "lon_max": 75,
        "lat_min": 5,
        "lat_max": 22
    }
}

YEARS = [2020, 2021, 2022, 2023, 2024]

PRES_MIN = 0
PRES_MAX = 1000


def download_argo_core_subset(basin_key, year):
    basin = BASINS[basin_key]

    start_date = f"{year}-01-01"
    end_date = f"{year + 1}-01-01"   # argopy uses max date as upper limit

    output_file = OUTPUT_DIR / f"ARGO_CORE_{basin_key}_{year}_0_1000dbar.nc"

    if output_file.exists():
        print("Already exists, skipping:", output_file.name)
        return output_file

    print("\nDownloading Argo Core:")
    print("Basin:", basin["name"])
    print("Year:", year)
    print("Output:", output_file)

    fetcher = ArgoDataFetcher().region([
        basin["lon_min"], basin["lon_max"],
        basin["lat_min"], basin["lat_max"],
        PRES_MIN, PRES_MAX,
        start_date, end_date
    ])

    ds = fetcher.to_xarray()

    ds.to_netcdf(output_file)

    print("Saved:", output_file)
    return output_file


for basin_key in BASINS:
    for year in YEARS:
        try:
            download_argo_core_subset(basin_key, year)
        except Exception as e:
            print("Failed:", basin_key, year)
            print("Reason:", e)

print("Argo Core subset download completed.")


Basin: Bay_of_Bengal
Year: 2020
Output: argo_profile_data/raw_core_argo/ARGO_CORE_bob_2020_0_1000dbar.nc
Saved: argo_profile_data/raw_core_argo/ARGO_CORE_bob_2020_0_1000dbar.nc

Basin: Bay_of_Bengal
Year: 2021
Output: argo_profile_data/raw_core_argo/ARGO_CORE_bob_2021_0_1000dbar.nc
Saved: argo_profile_data/raw_core_argo/ARGO_CORE_bob_2021_0_1000dbar.nc

Basin: Bay_of_Bengal
Year: 2022
Output: argo_profile_data/raw_core_argo/ARGO_CORE_bob_2022_0_1000dbar.nc
Saved: argo_profile_data/raw_core_argo/ARGO_CORE_bob_2022_0_1000dbar.nc

Basin: Bay_of_Bengal
Year: 2023
Output: argo_profile_data/raw_core_argo/ARGO_CORE_bob_2023_0_1000dbar.nc
Saved: argo_profile_data/raw_core_argo/ARGO_CORE_bob_2023_0_1000dbar.nc

Basin: Bay_of_Bengal
Year: 2024
Output: argo_profile_data/raw_core_argo/ARGO_CORE_bob_2024_0_1000dbar.nc
Saved: argo_profile_data/raw_core_argo/ARGO_CORE_bob_2024_0_1000dbar.nc

Basin: Arabian_Sea
Year: 2020
Output: argo_profile_data/raw_core_argo/ARGO_CORE_as_2020_0_1000dbar.nc
Saved: 

In [3]:
from pathlib import Path
import xarray as xr

raw_dir = Path("argo_profile_data/raw_core_argo")
files = sorted(raw_dir.glob("*.nc"))

print("Total Argo files:", len(files))

for f in files:
    print(f.name, round(f.stat().st_size / (1024 * 1024), 2), "MB")

# Inspect one file
sample_file = files[0]
ds = xr.open_dataset(sample_file)

print(ds)
print("Variables:", list(ds.data_vars))
print("Coordinates:", list(ds.coords))

ds.close()

Total Argo files: 10
ARGO_CORE_as_2020_0_1000dbar.nc 243.23 MB
ARGO_CORE_as_2021_0_1000dbar.nc 225.29 MB
ARGO_CORE_as_2022_0_1000dbar.nc 148.55 MB
ARGO_CORE_as_2023_0_1000dbar.nc 179.77 MB
ARGO_CORE_as_2024_0_1000dbar.nc 133.21 MB
ARGO_CORE_bob_2020_0_1000dbar.nc 82.92 MB
ARGO_CORE_bob_2021_0_1000dbar.nc 48.13 MB
ARGO_CORE_bob_2022_0_1000dbar.nc 31.57 MB
ARGO_CORE_bob_2023_0_1000dbar.nc 23.73 MB
ARGO_CORE_bob_2024_0_1000dbar.nc 34.24 MB
<xarray.Dataset> Size: 119MB
Dimensions:                   (N_POINTS: 662291)
Coordinates:
    LATITUDE                  (N_POINTS) float64 5MB ...
    LONGITUDE                 (N_POINTS) float64 5MB ...
    TIME                      (N_POINTS) datetime64[ns] 5MB ...
  * N_POINTS                  (N_POINTS) int64 5MB 0 1 2 ... 662289 662290
Data variables: (12/23)
    CONFIG_MISSION_NUMBER     (N_POINTS) float64 5MB ...
    CYCLE_NUMBER              (N_POINTS) float64 5MB ...
    DATA_MODE                 (N_POINTS) object 5MB ...
    DIRECTION        

In [4]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path

INPUT_DIR = Path("argo_profile_data/raw_core_argo")
OUTPUT_DIR = Path("argo_profile_data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def infer_basin_from_filename(filename):
    filename = filename.lower()
    if "_bob_" in filename:
        return "Bay_of_Bengal"
    elif "_as_" in filename:
        return "Arabian_Sea"
    return "Unknown"


def clean_qc_value(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x.startswith("b'") and x.endswith("'"):
        x = x[2:-1]
    return x.strip()


def process_one_argo_file(file_path):
    file_path = Path(file_path)
    basin = infer_basin_from_filename(file_path.name)

    print("\nProcessing:", file_path.name)

    ds = xr.open_dataset(file_path)

    df = ds.to_dataframe().reset_index()

    # Standardize column names
    df.columns = df.columns.str.lower()

    rename_map = {
        "time": "timestamp",
        "latitude": "latitude",
        "longitude": "longitude",
        "pres": "pressure_dbar",
        "temp": "temperature_c",
        "psal": "salinity_psu",
        "pres_qc": "pressure_qc",
        "temp_qc": "temperature_qc",
        "psal_qc": "salinity_qc",
        "platform_number": "platform_number",
        "cycle_number": "cycle_number"
    }

    existing_rename = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=existing_rename)

    required = [
        "timestamp",
        "latitude",
        "longitude",
        "pressure_dbar",
        "temperature_c",
        "salinity_psu"
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        print("Missing:", missing)
        print("Available:", df.columns.tolist())
        ds.close()
        raise ValueError("Required Argo columns missing.")

    df["basin"] = basin
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    # Convert numeric columns
    for col in ["latitude", "longitude", "pressure_dbar", "temperature_c", "salinity_psu"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Clean QC columns if available
    qc_cols = ["pressure_qc", "temperature_qc", "salinity_qc"]
    for col in qc_cols:
        if col in df.columns:
            df[col] = df[col].apply(clean_qc_value)

    # Keep only physically meaningful values
    df = df.dropna(subset=[
        "timestamp",
        "latitude",
        "longitude",
        "pressure_dbar",
        "temperature_c",
        "salinity_psu"
    ])

    df = df[
        (df["pressure_dbar"] >= 0) &
        (df["pressure_dbar"] <= 1000) &
        (df["temperature_c"] > -3) &
        (df["temperature_c"] < 40) &
        (df["salinity_psu"] > 0) &
        (df["salinity_psu"] < 45)
    ].copy()

    # QC filtering: keep good/probably good if QC is present
    good_qc = ["1", "2"]

    if "pressure_qc" in df.columns:
        df = df[df["pressure_qc"].isin(good_qc)]
    if "temperature_qc" in df.columns:
        df = df[df["temperature_qc"].isin(good_qc)]
    if "salinity_qc" in df.columns:
        df = df[df["salinity_qc"].isin(good_qc)]

    # Create profile ID
    if "platform_number" in df.columns and "cycle_number" in df.columns:
        df["profile_id"] = (
            df["platform_number"].astype(str) + "_" +
            df["cycle_number"].astype(str)
        )
    else:
        df["profile_id"] = (
            df["timestamp"].dt.strftime("%Y%m%d%H%M%S") + "_" +
            df["latitude"].round(3).astype(str) + "_" +
            df["longitude"].round(3).astype(str)
        )

    keep_cols = [
        "basin",
        "profile_id",
        "timestamp",
        "latitude",
        "longitude",
        "pressure_dbar",
        "temperature_c",
        "salinity_psu"
    ]

    for col in ["pressure_qc", "temperature_qc", "salinity_qc", "platform_number", "cycle_number"]:
        if col in df.columns:
            keep_cols.append(col)

    df = df[keep_cols].copy()

    ds.close()

    print("Rows:", len(df))
    return df


all_files = sorted(INPUT_DIR.glob("*.nc"))

frames = []

for file in all_files:
    try:
        frames.append(process_one_argo_file(file))
    except Exception as e:
        print("Skipped:", file.name)
        print("Reason:", e)

if len(frames) == 0:
    raise ValueError("No Argo files processed successfully.")

argo_points = pd.concat(frames, ignore_index=True)

print("\nFinal Argo point-level shape:", argo_points.shape)
argo_points.head()


Processing: ARGO_CORE_as_2020_0_1000dbar.nc
Rows: 427740

Processing: ARGO_CORE_as_2021_0_1000dbar.nc
Rows: 381714

Processing: ARGO_CORE_as_2022_0_1000dbar.nc
Rows: 271430

Processing: ARGO_CORE_as_2023_0_1000dbar.nc
Rows: 369860

Processing: ARGO_CORE_as_2024_0_1000dbar.nc
Rows: 328527

Processing: ARGO_CORE_bob_2020_0_1000dbar.nc
Rows: 190613

Processing: ARGO_CORE_bob_2021_0_1000dbar.nc
Rows: 129009

Processing: ARGO_CORE_bob_2022_0_1000dbar.nc
Rows: 96449

Processing: ARGO_CORE_bob_2023_0_1000dbar.nc
Rows: 49319

Processing: ARGO_CORE_bob_2024_0_1000dbar.nc
Rows: 82662

Final Argo point-level shape: (2327323, 13)


,basin,profile_id,timestamp,latitude,longitude,pressure_dbar,temperature_c,salinity_psu,pressure_qc,temperature_qc,salinity_qc,platform_number,cycle_number
0,Arabian_Sea,2901857_234.0,2020-01-01 01:51:52,11.415,66.628,2.6,28.681999,36.111,1,1,1,2901857,234.0
1,Arabian_Sea,2901857_234.0,2020-01-01 01:51:52,11.415,66.628,4.0,28.684000,36.111,1,1,1,2901857,234.0
2,Arabian_Sea,2901857_234.0,2020-01-01 01:51:52,11.415,66.628,6.0,28.684999,36.111,1,1,1,2901857,234.0
3,Arabian_Sea,2901857_234.0,2020-01-01 01:51:52,11.415,66.628,8.0,28.686001,36.111,1,1,1,2901857,234.0
4,Arabian_Sea,2901857_234.0,2020-01-01 01:51:52,11.415,66.628,10.0,28.686001,36.111,1,1,1,2901857,234.0


In [5]:
point_csv = OUTPUT_DIR / "argo_core_points_bob_as_2020_2024.csv"
point_parquet = OUTPUT_DIR / "argo_core_points_bob_as_2020_2024.parquet"

argo_points.to_csv(point_csv, index=False)
argo_points.to_parquet(point_parquet, index=False)

print("Saved CSV:", point_csv)
print("Saved Parquet:", point_parquet)

Saved CSV: argo_profile_data/processed/argo_core_points_bob_as_2020_2024.csv
Saved Parquet: argo_profile_data/processed/argo_core_points_bob_as_2020_2024.parquet


In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

STANDARD_DEPTHS = np.array([0, 10, 20, 30, 50, 75, 100, 150, 200, 300, 500, 700, 1000], dtype=float)

OUTPUT_DIR = Path("argo_profile_data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def interpolate_one_profile(group):
    group = group.sort_values("pressure_dbar")

    group = group.dropna(subset=["pressure_dbar", "temperature_c", "salinity_psu"])

    # Remove duplicate pressure levels
    group = group.drop_duplicates(subset=["pressure_dbar"])

    if len(group) < 3:
        return pd.DataFrame()

    pres = group["pressure_dbar"].values
    temp = group["temperature_c"].values
    psal = group["salinity_psu"].values

    min_p = np.nanmin(pres)
    max_p = np.nanmax(pres)

    valid_depths = STANDARD_DEPTHS[
        (STANDARD_DEPTHS >= min_p) &
        (STANDARD_DEPTHS <= max_p)
    ]

    if len(valid_depths) == 0:
        return pd.DataFrame()

    temp_interp = np.interp(valid_depths, pres, temp)
    psal_interp = np.interp(valid_depths, pres, psal)

    first = group.iloc[0]

    out = pd.DataFrame({
        "basin": first["basin"],
        "profile_id": first["profile_id"],
        "timestamp": first["timestamp"],
        "latitude": first["latitude"],
        "longitude": first["longitude"],
        "depth_m": valid_depths,
        "temperature_c": temp_interp,
        "salinity_psu": psal_interp,
        "source": "Argo_Core"
    })

    return out


profile_frames = []

group_cols = ["basin", "profile_id"]

for _, group in argo_points.groupby(group_cols):
    result = interpolate_one_profile(group)
    if not result.empty:
        profile_frames.append(result)

argo_standard_profiles = pd.concat(profile_frames, ignore_index=True)

print("Standard-depth profile table shape:", argo_standard_profiles.shape)
argo_standard_profiles.head()

Standard-depth profile table shape: (98389, 9)


,basin,profile_id,timestamp,latitude,longitude,depth_m,temperature_c,salinity_psu,source
0,Arabian_Sea,1901739_100.0,2020-07-02 15:30:53,13.558,61.033,10.0,28.496000,35.959000,Argo_Core
1,Arabian_Sea,1901739_100.0,2020-07-02 15:30:53,13.558,61.033,20.0,28.497999,35.957001,Argo_Core
2,Arabian_Sea,1901739_100.0,2020-07-02 15:30:53,13.558,61.033,30.0,28.499001,35.955002,Argo_Core
3,Arabian_Sea,1901739_100.0,2020-07-02 15:30:53,13.558,61.033,50.0,28.488001,35.985001,Argo_Core
4,Arabian_Sea,1901739_100.0,2020-07-02 15:30:53,13.558,61.033,75.0,25.231000,36.084499,Argo_Core


In [7]:
standard_csv = OUTPUT_DIR / "argo_core_standard_depth_profiles_bob_as_2020_2024.csv"
standard_excel = OUTPUT_DIR / "argo_core_standard_depth_profiles_sample.xlsx"
standard_parquet = OUTPUT_DIR / "argo_core_standard_depth_profiles_bob_as_2020_2024.parquet"

argo_standard_profiles.to_csv(standard_csv, index=False)
argo_standard_profiles.to_parquet(standard_parquet, index=False)

# Excel sample only, because full file may be large
argo_standard_profiles.head(100000).to_excel(standard_excel, index=False)

print("Saved full CSV:", standard_csv)
print("Saved full Parquet:", standard_parquet)
print("Saved Excel sample:", standard_excel)

Saved full CSV: argo_profile_data/processed/argo_core_standard_depth_profiles_bob_as_2020_2024.csv
Saved full Parquet: argo_profile_data/processed/argo_core_standard_depth_profiles_bob_as_2020_2024.parquet
Saved Excel sample: argo_profile_data/processed/argo_core_standard_depth_profiles_sample.xlsx


In [8]:
argo_standard_profiles["date"] = pd.to_datetime(argo_standard_profiles["timestamp"]).dt.date

argo_daily_summary = (
    argo_standard_profiles.groupby(["basin", "date", "depth_m"], as_index=False)
    .agg(
        temperature_mean=("temperature_c", "mean"),
        temperature_min=("temperature_c", "min"),
        temperature_max=("temperature_c", "max"),
        salinity_mean=("salinity_psu", "mean"),
        salinity_min=("salinity_psu", "min"),
        salinity_max=("salinity_psu", "max"),
        profile_count=("profile_id", "nunique"),
        observation_count=("temperature_c", "count")
    )
)

summary_csv = OUTPUT_DIR / "argo_core_daily_depth_summary_bob_as_2020_2024.csv"
summary_excel = OUTPUT_DIR / "argo_core_daily_depth_summary_bob_as_2020_2024.xlsx"
summary_parquet = OUTPUT_DIR / "argo_core_daily_depth_summary_bob_as_2020_2024.parquet"

argo_daily_summary.to_csv(summary_csv, index=False)
argo_daily_summary.to_excel(summary_excel, index=False)
argo_daily_summary.to_parquet(summary_parquet, index=False)

print("Daily summary shape:", argo_daily_summary.shape)
print("Saved Excel:", summary_excel)

argo_daily_summary.head()

Daily summary shape: (35592, 11)
Saved Excel: argo_profile_data/processed/argo_core_daily_depth_summary_bob_as_2020_2024.xlsx


,basin,date,depth_m,temperature_mean,temperature_min,temperature_max,salinity_mean,salinity_min,salinity_max,profile_count,observation_count
0,Arabian_Sea,2020-01-01,10.0,28.044915,26.483231,29.183650,35.713707,34.957750,36.111000,4,4
1,Arabian_Sea,2020-01-01,20.0,28.035288,26.468778,29.170800,35.728715,34.995054,36.111000,4,4
2,Arabian_Sea,2020-01-01,30.0,28.063913,26.499001,29.135593,35.791529,34.998939,36.334999,4,4
3,Arabian_Sea,2020-01-01,50.0,27.953216,26.514928,29.033001,36.111834,35.823315,36.632999,4,4
4,Arabian_Sea,2020-01-01,75.0,27.237309,26.049000,28.381162,36.338547,35.987999,36.583000,4,4


In [9]:
import shutil
from google.colab import files

zip_name = "argo_core_profiles_bob_as_2020_2024"

shutil.make_archive(zip_name, "zip", "argo_profile_data")

files.download(zip_name + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>